In [11]:
import pandas as pd
from elasticsearch import Elasticsearch, helpers
from tqdm.auto import tqdm
import time

es = Elasticsearch("http://localhost:9200")
index_name = "recipes"

index_config = {
    "settings": {
        "index": {
            "number_of_shards": 1,
            "number_of_replicas": 0,
            "refresh_interval": "-1",
            "max_ngram_diff": 20
        },
        "analysis": {
            "filter": {
                "autocomplete_filter": {
                    "type": "edge_ngram",
                    "min_gram": 2,
                    "max_gram": 20
                },
                "shingle_filter": {
                    "type": "shingle",
                    "min_shingle_size": 2,
                    "max_shingle_size": 3,
                    "output_unigrams": True
                }
            },
            "tokenizer": {
                "ngram_tokenizer": {
                    "type": "ngram",
                    "min_gram": 3,
                    "max_gram": 10,
                    "token_chars": ["letter", "digit"]
                }
            },
            "analyzer": {
                "recipe_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "stop", "porter_stem"]
                },
                "autocomplete_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "autocomplete_filter"]
                },
                "shingle_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase", "shingle_filter"]
                },
                "ngram_analyzer": {
                    "type": "custom",
                    "tokenizer": "ngram_tokenizer",
                    "filter": ["lowercase"]
                },
                "search_analyzer": {
                    "type": "custom",
                    "tokenizer": "standard",
                    "filter": ["lowercase"]
                }
            }
        }
    },
    "mappings": {
        "properties": {
            "RecipeId": {"type": "integer"},
            "Name": {
                "type": "text",
                "analyzer": "recipe_analyzer",
                "fields": {
                    "autocomplete": {
                        "type": "text",
                        "analyzer": "autocomplete_analyzer",
                        "search_analyzer": "search_analyzer"
                    },
                    "shingle": {
                        "type": "text",
                        "analyzer": "shingle_analyzer"
                    },
                    "ngram": {
                        "type": "text",
                        "analyzer": "ngram_analyzer",
                        "search_analyzer": "search_analyzer"
                    },
                    "english": {
                        "type": "text",
                        "analyzer": "english"
                    }
                }
            },
            "NameSuggest": {"type": "completion"},
            "RecipeIngredientParts": {
                "type": "text",
                "analyzer": "recipe_analyzer",
                "term_vector": "yes"
            },
            "Description": {"type": "text", "analyzer": "recipe_analyzer"},
            "RecipeInstructions": {"type": "text", "analyzer": "recipe_analyzer"},
            "Keywords": {"type": "text", "analyzer": "recipe_analyzer"},
            "RecipeCategory": {
                "type": "keyword",
                "fields": {
                    "text": {
                        "type": "text",
                        "analyzer": "standard"
                    }
                }
            },
            "Images": {"type": "keyword", "index": False},
            "AggregatedRating": {"type": "float"},
            "Calories": {"type": "float"},
            "CookTimeMins": {"type": "integer"},
            "PrepTimeMins": {"type": "integer"},
            "TotalTimeMins": {"type": "integer"}
        }
    }
}

if es.indices.exists(index=index_name):
    es.indices.delete(index=index_name)

es.indices.create(index=index_name, settings=index_config["settings"], mappings=index_config["mappings"])

df = pd.read_csv('../data/recipes_ready_for_es.csv')
df = df.drop_duplicates(subset=["RecipeId"])

def clean_data(row):
    for key, value in row.items():
        if pd.isna(value):
            row[key] = None

    float_fields = ['AggregatedRating', 'Calories']
    int_fields = ['CookTimeMins', 'PrepTimeMins', 'TotalTimeMins', 'RecipeId']

    for f in float_fields:
        row[f] = float(row[f]) if row.get(f) is not None and row[f] != "" else 0.0

    for f in int_fields:
        row[f] = int(float(row[f])) if row.get(f) is not None and row[f] != "" else 0

    if isinstance(row.get('Images'), str) and len(row['Images']) > 10000:
        row['Images'] = row['Images'][:10000]

    if isinstance(row.get('RecipeCategory'), str) and len(row['RecipeCategory']) > 10000:
        row['RecipeCategory'] = row['RecipeCategory'][:10000]

    if row.get("Name") and isinstance(row.get("Name"), str):
        row["NameSuggest"] = {"input": row["Name"]}

    return row

def generate_docs(dataframe):
    for row in dataframe.to_dict(orient="records"):
        cleaned_row = clean_data(row)
        yield {
            "_index": index_name,
            "_id": cleaned_row["RecipeId"],
            "_source": cleaned_row
        }

start_time = time.time()

success = 0
failed = 0

with tqdm(total=len(df), desc="Indexing", unit="docs") as pbar:
    for ok, result in helpers.streaming_bulk(
        client=es,
        actions=generate_docs(df),
        chunk_size=2000,
        request_timeout=120,
        raise_on_error=False
    ):
        if ok:
            success += 1
        else:
            failed += 1
        pbar.update(1)

end_time = time.time()

es.indices.put_settings(index=index_name, body={"index": {"refresh_interval": "1s"}})

count = es.count(index=index_name)["count"]

print(f"Done in {end_time - start_time:.2f}s")
print(f"Success: {success}")
print(f"Failed: {failed}")
print(f"Docs: {count}")

Indexing:   0%|          | 0/522517 [00:00<?, ?docs/s]/var/folders/kv/6w4vv49n0lv80zcs_s67jzz40000gn/T/ipykernel_10380/1192516321.py:171: DeprecationWarning: Passing transport options in the API method is deprecated. Use 'Elasticsearch.options()' instead.
  for ok, result in helpers.streaming_bulk(
Indexing: 100%|██████████| 522517/522517 [04:38<00:00, 1878.85docs/s]


Done in 278.25s
Success: 522517
Failed: 0
Docs: 0


In [23]:
from elasticsearch import Elasticsearch

es = Elasticsearch("http://localhost:9200")
index_name = "recipes"

search_text = "Green curry"

search_body = {
    "size": 5,
    "_source": [
        "Name",
        "TotalTimeMins",
        "RecipeCategory",
        "RecipeIngredientParts"
    ],
    "query": {
        "multi_match": {
            "query": search_text,
            "type": "most_fields",
            "fields": [
                "Name^10",
                "Name.shingle^5",
                "Name.english^3",
                "Name.ngram^1",
                "Keywords^2"
            ],
            "fuzziness": "AUTO"
        }
    }
}

try:
    response = es.search(index=index_name, body=search_body)

    hits = response["hits"]["hits"]

    for idx, hit in enumerate(hits, 1):
        source = hit["_source"]
        score = round(hit["_score"], 2) if hit["_score"] else 0

        print(f"Rank {idx} | Score: {score}")
        print(f"Recipe name: {source.get('Name')}")
        print(f"Total time: {source.get('TotalTimeMins')} minutes")
        print(f"Category: {source.get('RecipeCategory')}")

        ingredients = source.get("RecipeIngredientParts")
        if isinstance(ingredients, list):
            ingredients_text = ", ".join(ingredients)
        else:
            ingredients_text = ingredients

        print(f"Ingredients: {ingredients_text}")
        print("-" * 50)

except Exception as e:
    print(f"Error: {e}")

Rank 1 | Score: 249.02
Recipe name: Green Curry
Total time: 60 minutes
Category: Curries
Ingredients: potato, green beans, zucchini, spich, butter, onions, garlic cloves, turmeric, cumin, hot paprika, cayenne pepper, black pepper, cinmon, salt, fresh ginger, green chilies, lemon juice, water
--------------------------------------------------
Rank 2 | Score: 227.21
Recipe name: Easy Green Curry
Total time: 30 minutes
Category: Curries
Ingredients: kaffir lime leaves, garlic cloves, coconut milk, green curry paste, fish sauce, sugar, chicken breast, Japanese eggplants, bamboo shoots, green beans, Thai basil, rice
--------------------------------------------------
Rank 3 | Score: 227.21
Recipe name: Thai Green Curry
Total time: 40 minutes
Category: Curries
Ingredients: onion, garlic cloves, green curry paste, olive oil, chicken breast, firm tofu, fish sauce, light coconut milk, kaffir lime leaves, courgette, fresh basil leaf
--------------------------------------------------
Rank 4 | Scor

In [1]:
import pandas as pd
import ast
from collections import Counter

print("Loading data...")
df = pd.read_csv('../data/recipes_ready_for_es.csv')

def get_top_keywords(df, top_n=30):
    all_keywords = []

    for keywords_str in df['Keywords'].dropna():
        try:

            if isinstance(keywords_str, str) and keywords_str.startswith('['):
                kw_list = ast.literal_eval(keywords_str)
                all_keywords.extend(kw_list)
            else:
                kw_list = [k.strip() for k in keywords_str.split(',')]
                all_keywords.extend(kw_list)
        except:
            continue

    counter = Counter(all_keywords)
    return counter.most_common(top_n)

print("\n Top 30 Keywords:")
top_30 = get_top_keywords(df, 30)
for word, count in top_30:
    print(f"- '{word}' (เจอ {count} สูตร)")

print("\n Top 20 Categories:")
print(df['RecipeCategory'].value_counts().head(20))

Loading data...

 Top 30 Keywords:
- 'Easy' (เจอ 276838 สูตร)
- '< 60 Mins' (เจอ 149589 สูตร)
- '< 30 Mins' (เจอ 112229 สูตร)
- '< 4 Hours' (เจอ 111071 สูตร)
- 'Meat' (เจอ 103191 สูตร)
- '< 15 Mins' (เจอ 89340 สูตร)
- 'Healthy' (เจอ 83305 สูตร)
- 'Vegetable' (เจอ 81264 สูตร)
- 'Low Cholesterol' (เจอ 74121 สูตร)
- 'Beginner Cook' (เจอ 67092 สูตร)
- 'Inexpensive' (เจอ 66052 สูตร)
- 'Low Protein' (เจอ 63972 สูตร)
- 'Fruit' (เจอ 60523 สูตร)
- 'Oven' (เจอ 54504 สูตร)
- 'Kid Friendly' (เจอ 51296 สูตร)
- 'European' (เจอ 47121 สูตร)
- 'Poultry' (เจอ 46278 สูตร)
- 'Weeknight' (เจอ 44914 สูตร)
- 'Dessert' (เจอ 43342 สูตร)
- 'For Large Groups' (เจอ 41337 สูตร)
- 'Stove Top' (เจอ 38254 สูตร)
- 'Brunch' (เจอ 36278 สูตร)
- 'Chicken' (เจอ 28312 สูตร)
- 'Cookie & Brownie' (เจอ 28039 สูตร)
- 'Asian' (เจอ 25424 สูตร)
- 'Free Of...' (เจอ 23158 สูตร)
- 'High In...' (เจอ 22127 สูตร)
- 'Sweet' (เจอ 21594 สูตร)
- 'Cheese' (เจอ 20137 สูตร)
- 'Savory' (เจอ 20028 สูตร)

 Top 20 Categories:
RecipeCategory
Desser

In [ ]:
df.columns

In [3]:
import pandas as pd
import ast
from collections import Counter

print("Loading data...")
df = pd.read_csv('../data/recipes_ready_for_es.csv')

def get_top_keywords(df, top_n=30):
    all_keywords = []

    for keywords_str in df['Keywords'].dropna():
        try:

            if isinstance(keywords_str, str) and keywords_str.startswith('['):
                kw_list = ast.literal_eval(keywords_str)
                all_keywords.extend(kw_list)
            else:
                kw_list = [k.strip() for k in keywords_str.split(',')]
                all_keywords.extend(kw_list)
        except:
            continue

    counter = Counter(all_keywords)
    return counter.most_common(top_n)

print("\n Top 30 Keywords:")
top_30 = get_top_keywords(df, 30)
for word, count in top_30:
    print(f"- '{word}' (เจอ {count} สูตร)")

print("\n️ Top 20 Categories:")
print(df['RecipeCategory'].value_counts().head(20))

Loading data...

 Top 30 Keywords:
- 'Easy' (เจอ 276838 สูตร)
- '< 60 Mins' (เจอ 149589 สูตร)
- '< 30 Mins' (เจอ 112229 สูตร)
- '< 4 Hours' (เจอ 111071 สูตร)
- 'Meat' (เจอ 103191 สูตร)
- '< 15 Mins' (เจอ 89340 สูตร)
- 'Healthy' (เจอ 83305 สูตร)
- 'Vegetable' (เจอ 81264 สูตร)
- 'Low Cholesterol' (เจอ 74121 สูตร)
- 'Beginner Cook' (เจอ 67092 สูตร)
- 'Inexpensive' (เจอ 66052 สูตร)
- 'Low Protein' (เจอ 63972 สูตร)
- 'Fruit' (เจอ 60523 สูตร)
- 'Oven' (เจอ 54504 สูตร)
- 'Kid Friendly' (เจอ 51296 สูตร)
- 'European' (เจอ 47121 สูตร)
- 'Poultry' (เจอ 46278 สูตร)
- 'Weeknight' (เจอ 44914 สูตร)
- 'Dessert' (เจอ 43342 สูตร)
- 'For Large Groups' (เจอ 41337 สูตร)
- 'Stove Top' (เจอ 38254 สูตร)
- 'Brunch' (เจอ 36278 สูตร)
- 'Chicken' (เจอ 28312 สูตร)
- 'Cookie & Brownie' (เจอ 28039 สูตร)
- 'Asian' (เจอ 25424 สูตร)
- 'Free Of...' (เจอ 23158 สูตร)
- 'High In...' (เจอ 22127 สูตร)
- 'Sweet' (เจอ 21594 สูตร)
- 'Cheese' (เจอ 20137 สูตร)
- 'Savory' (เจอ 20028 สูตร)

️ Top 20 Categories:
RecipeCategory
Desse

In [4]:
df.columns

Index(['RecipeId', 'Name', 'Description', 'Images', 'RecipeCategory',
       'Keywords', 'RecipeIngredientParts', 'AggregatedRating', 'Calories',
       'FatContent', 'SaturatedFatContent', 'CholesterolContent',
       'SodiumContent', 'CarbohydrateContent', 'FiberContent', 'SugarContent',
       'ProteinContent', 'RecipeInstructions', 'CookTimeMins', 'PrepTimeMins',
       'TotalTimeMins'],
      dtype='object')